# incident_telemetry_stitch

**What it does:** for each network incident, summarise what the interfaces on that
device looked like in the hour before it opened.

**Owner:** was Raghav, then me, now unclear
**Runs:** manually, when someone asks. Takes about an hour end to end.
**Output:** `SCRATCH.INCIDENT_TELEMETRY` — a few people query it directly

> Known issues:
>   - cmdb_ci is a mess, we only catch the clean ones
>   - has to be re-run in full every time
>   - if the numbers look wrong, check the join before believing them


## Setup

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()
session.sql("USE DATABASE ENS_SANDBOX").collect()
session.sql("USE SCHEMA RAW").collect()

LOOKBACK_HOURS = 1        # changed this to 2 once for an analysis, changed it back. probably.
SEVERITY_FILTER = ["S1", "S2"]

## Resolve incidents to devices

Join ServiceNow's `cmdb_ci` to the device inventory. This is the part that eats the hour.

In [ ]:
incidents = session.table("ENS_INCIDENT").filter(F.col("SEVERITY").isin(SEVERITY_FILTER))
devices   = session.table("ENS_DEVICE").select("DEVICE_ID", "HOSTNAME", "FQDN", "SITE_CODE")

# cmdb_ci comes in as: exact hostname, mixed case, fqdn, "SITE / DEV-nnnnnn", or free text.
# we only handle the first two. the rest fall on the floor.
resolved = (
    incidents
      .with_column("CI_CLEAN", F.lower(F.trim(F.col("CMDB_CI"))))
      .join(devices, F.col("CI_CLEAN") == devices["HOSTNAME"], how="inner")
      .select(
          "INCIDENT_NUMBER", "OPENED_TS", "RESOLVED_TS", "SEVERITY",
          "ASSIGNMENT_GROUP", "DEVICE_ID", "SITE_CODE",
      )
)

print("incidents in:", incidents.count(), "resolved:", resolved.count())
# ^ this ratio is bad and everyone knows it

## Pull the hour before each incident

In [ ]:
ifaces = session.table("ENS_INTERFACE").filter(F.col("IS_MONITORED"))
metrics = session.table("ENS_INTERFACE_METRIC_5M")

resolved.create_or_replace_temp_view("RESOLVED_INCIDENTS")
ifaces.create_or_replace_temp_view("MONITORED_IFACES")

# interval join, written as SQL because expressing it in Snowpark was worse
precursor = session.sql(f"""
    select
        r.INCIDENT_NUMBER,
        avg(m.UTILIZATION_PCT_OUT)          as AVG_UTIL_OUT,
        max(m.UTILIZATION_PCT_OUT)          as PEAK_UTIL_OUT,
        sum(m.IN_ERRORS + m.OUT_ERRORS)     as ERRORS,
        max(m.LATENCY_MS_P95)               as PEAK_LATENCY_P95,
        max(m.PACKET_LOSS_PCT)              as PEAK_LOSS,
        count_if(m.POLL_STATUS <> 'ok')     as FAILED_POLLS
    from RESOLVED_INCIDENTS r
    join MONITORED_IFACES i
      on i.DEVICE_ID = r.DEVICE_ID
    join ENS_INTERFACE_METRIC_5M m
      on  m.INTERFACE_ID = i.INTERFACE_ID
      and m.METRIC_TS >= dateadd('hour', -{LOOKBACK_HOURS}, r.OPENED_TS)
      and m.METRIC_TS <  r.OPENED_TS
    group by 1
""")

precursor.create_or_replace_temp_view("PRECURSOR")

## Stitch and write

In [ ]:
final = session.sql("""
    select
        r.*,
        p.AVG_UTIL_OUT,
        p.PEAK_UTIL_OUT,
        p.ERRORS,
        p.PEAK_LATENCY_P95,
        p.PEAK_LOSS,
        p.FAILED_POLLS,
        datediff('minute', r.OPENED_TS, r.RESOLVED_TS) as MINUTES_TO_RESOLVE
    from RESOLVED_INCIDENTS r
    left join PRECURSOR p on p.INCIDENT_NUMBER = r.INCIDENT_NUMBER
""")

session.sql("DROP TABLE IF EXISTS SCRATCH.INCIDENT_TELEMETRY").collect()
final.write.mode("overwrite").save_as_table("SCRATCH.INCIDENT_TELEMETRY")

print("rows written:", session.table("SCRATCH.INCIDENT_TELEMETRY").count())

## Sanity check

In [ ]:
nulls = session.sql("""
    select count(*) as C from SCRATCH.INCIDENT_TELEMETRY where PEAK_UTIL_OUT is null
""").collect()[0]["C"]

print("incidents with no telemetry:", nulls)
# sometimes this is high. sometimes that's real (device not monitored),
# sometimes the join broke. no way to tell from here.

# TODO: check incident_number is unique
# TODO: check utilization is 0-100
# TODO: figure out what's downstream of this before changing it